# UMI in MuJoCo - Phase 6: Run the trained policy in the simulator

Set **Runtime > Change runtime type > GPU**. Run the cells in order.

**What this does:** this is the simulated version of UMI's `eval_real.py`. Every 0.1 s the loop
1. takes the wrist-camera image and gripper pose (last 2 frames), exactly as the policy was trained,
2. asks the policy for 16 future gripper poses and finger widths (relative to the gripper's current pose),
3. converts them to world poses with the repo's own `convert_pose_mat_rep(..., backward=True)`,
4. executes the first 8 of them through inverse kinematics on the simulated Panda, then repeats (receding horizon).

The cup counts as placed when it sits on the saucer, upright, with the gripper open, for 1 s.

**One simulator-specific rule:** the policy predicts the *measured* finger width. While holding the cup this is about 5 cm (the cup's diameter), and a gripper told to go to 5 cm would not squeeze. So any predicted width below 6.5 cm is sent to the gripper as "fully closed". On a real UMI gripper the same job is done by the force-limited close.

**Order of the steps:** the **oracle test** (step 7) comes first. It replaces the policy by a perfect one that reads its plan from a recorded demo. If that fails, something *around* the policy is broken (controller, conversion, success check), and no amount of training will help. Only then is the learned policy tested (steps 8-9).

**I have not been able to run MuJoCo here.** Everything except the physics (conversions, controller logic, episode loop, failure handling) was tested offline against the repo's own pose code and a mock simulator.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Paths, Google Drive, and the Phase 2 scene code

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/umi_mujoco'
REPO = '/content/universal_manipulation_interface'
ENV  = '/content/umi_train_env'
PY   = f'{ENV}/bin/python'
DEMO_DIR = f'{ROOT}/sim_demos'
os.environ.update(ROOT=ROOT, REPO=REPO, ENV=ENV, PY=PY, DEMO_DIR=DEMO_DIR, WANDB_MODE='disabled')

src = f'{ROOT}/sim/umi_sim'
assert os.path.exists(f'{src}/scene.py') and os.path.exists(f'{src}/env.py'), 'Phase 2 code not found in ' + src
shutil.copytree(src, '/content/umi_sim', dirs_exist_ok=True)
demos = sorted(f for f in os.listdir(DEMO_DIR) if f.startswith('ep_')) if os.path.isdir(DEMO_DIR) else []
print('scene code restored:', sorted(os.listdir('/content/umi_sim')))
print('demos on Drive:', len(demos), '(needed only for the oracle test)')
ckpts = []
for run in sorted(os.listdir(f'{ROOT}/checkpoints')) if os.path.isdir(f'{ROOT}/checkpoints') else []:
    d = f'{ROOT}/checkpoints/{run}/checkpoints'
    if os.path.isdir(d):
        ckpts += [f'{d}/{f}' for f in sorted(os.listdir(d))]
print('checkpoints found:'); [print('  ', c) for c in ckpts]

## 3. Environment: system libraries, Python 3.9 env (same as Phase 5) plus MuJoCo, repo, scene assets
About 6-8 minutes in a fresh session.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libglfw3 libosmesa6-dev patchelf ffmpeg > /dev/null
if [ ! -d "$REPO/.git" ]; then
  git clone https://github.com/real-stanford/universal_manipulation_interface.git "$REPO"
fi
MEN=/content/mujoco_menagerie
if [ ! -d "$MEN/franka_emika_panda" ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie.git "$MEN"
  cd "$MEN" && git sparse-checkout set franka_emika_panda
fi
echo "repo + menagerie ready"

In [ ]:
%%writefile /content/umi_train_requirements.txt
torch==2.1.0
torchvision==0.16.0
numpy==1.24.4
scipy==1.11.4
numba==0.57.1
pandas==2.1.4
opencv-python-headless==4.7.0.72
matplotlib==3.7.5
av==12.3.0
pyyaml==6.0.1
tqdm==4.65.0
zarr==2.16.1
numcodecs==0.11.0
hydra-core==1.2.0
wandb==0.15.8
dill==0.3.7
einops==0.6.1
diffusers==0.18.2
huggingface_hub==0.19.4
scikit-video==1.1.11
lmdb
timm==0.9.7
scikit-fda==0.9
cython==0.29.37
accelerate==0.24.1
boto3
aiohttp==3.9.5
aiofiles==23.2.1
threadpoolctl==3.2.0
imagecodecs==2023.9.18
h5py
psutil
termcolor
tensorboardX
imageio
imageio-ffmpeg
filelock

In [ ]:
%%bash
set -eo pipefail
pip -q install uv
if [ -x "$PY" ]; then
  echo "environment already exists (re-using)"
else
  uv python install 3.9
  uv venv "$ENV" --python 3.9 --seed
fi
uv pip install --python "$PY" -r /content/umi_train_requirements.txt 2>&1 | tail -8
uv pip install --python "$PY" --no-deps robomimic==0.2.0 2>&1 | tail -2
uv pip install --python "$PY" mujoco==3.2.3 imageio imageio-ffmpeg 2>&1 | tail -2
"$PY" -c "import torch, mujoco; print('torch', torch.__version__, '| gpu', torch.cuda.is_available(), '| mujoco', mujoco.__version__)"

## 4. Find a working OpenGL backend (for the simulated camera)

In [ ]:
%%bash
cat > /content/gl_test.py <<'EOF'
import mujoco
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size="1"/></worldbody></mujoco>')
d = mujoco.MjData(m)
r = mujoco.Renderer(m, 64, 64)
mujoco.mj_forward(m, d)
r.update_scene(d)
r.render()
print('ok')
EOF
rm -f /content/gl_backend.txt
for GL in egl osmesa; do
  if MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL "$PY" /content/gl_test.py > /dev/null 2>&1; then
    echo $GL > /content/gl_backend.txt
    break
  fi
done
if [ -f /content/gl_backend.txt ]; then echo "working OpenGL backend: $(cat /content/gl_backend.txt)"; else echo "NO WORKING GL BACKEND"; fi

## 5. Write the evaluation code
`sim_ctrl.py` is a copy of the kinematics and IK code the Phase 3 expert used. `run_policy.py` is the evaluation loop described above.

In [ ]:
%%writefile /content/umi_sim/sim_ctrl.py
"""Kinematics and inverse kinematics for the simulated Panda (identical to the code the Phase 3 expert used)."""
import numpy as np
from scipy.spatial.transform import Rotation as R


class MjKinematics:
    """Forward kinematics and Jacobian of the TCP site for the 7 arm joints, using a private MjData."""

    def __init__(self, model, tcp_site_id):
        import mujoco
        self.mj = mujoco
        self.model = model
        self.data = mujoco.MjData(model)
        hinge = int(mujoco.mjtJoint.mjJNT_HINGE)
        jids = [j for j in range(model.njnt) if int(model.jnt_type[j]) == hinge]
        if len(jids) != 7:
            raise RuntimeError("expected 7 hinge joints, found %d" % len(jids))
        self.qadr = np.array([model.jnt_qposadr[j] for j in jids])
        self.dadr = np.array([model.jnt_dofadr[j] for j in jids])
        self.lo = np.array(model.jnt_range[jids, 0], dtype=float)
        self.hi = np.array(model.jnt_range[jids, 1], dtype=float)
        self.site = tcp_site_id
        self._jp = np.zeros((3, model.nv))
        self._jr = np.zeros((3, model.nv))

    def fk(self, q):
        m, d = self.model, self.data
        d.qpos[self.qadr] = q
        self.mj.mj_kinematics(m, d)
        self.mj.mj_comPos(m, d)
        self.mj.mj_jacSite(m, d, self._jp, self._jr, self.site)
        pos = d.site_xpos[self.site].copy()
        rot = d.site_xmat[self.site].reshape(3, 3).copy()
        jac = np.vstack([self._jp[:, self.dadr], self._jr[:, self.dadr]])
        return pos, rot, jac


def ik_step(kin, q, pos_des, rot_des, q_home, iters=2, lam=0.05, max_dq=0.05, null_gain=0.1):
    """A few damped-least-squares iterations towards the desired TCP pose.
    The null-space term keeps the elbow close to the ready posture."""
    q = np.asarray(q, dtype=float).copy()
    for _ in range(iters):
        pos, rot, jac = kin.fk(q)
        err = np.concatenate([pos_des - pos, R.from_matrix(rot_des @ rot.T).as_rotvec()])
        jpinv = jac.T @ np.linalg.inv(jac @ jac.T + lam ** 2 * np.eye(6))
        dq = jpinv @ err + (np.eye(len(q)) - jpinv @ jac) @ (null_gain * (q_home - q))
        peak = np.abs(dq).max()
        if peak > max_dq:
            dq *= max_dq / peak
        q = np.clip(q + dq, kin.lo, kin.hi)
    return q

In [ ]:
%%writefile /content/umi_sim/run_policy.py
"""Run a trained UMI policy in the MuJoCo cup-on-saucer scene: the simulated counterpart of eval_real.py.

Every 0.1 s ("tick", the same rate as the demos) the loop
  1. builds the observation exactly like get_real_umi_obs_dict (images / 255, poses relative to the newest frame,
     rotation as 6-D, rotation relative to the episode start),
  2. asks the policy for 16 future relative poses + gripper widths,
  3. turns them into world poses relative to the CURRENT measured pose (get_real_umi_action),
  4. executes the first `steps_per_inference` of them (receding horizon), one waypoint per tick, through
     damped-least-squares IK and the Panda joint position servos.

    python run_policy.py --menagerie /content/mujoco_menagerie --ckpt latest.ckpt --out /content/eval --n 10
    python run_policy.py --menagerie ... --oracle --demo_dir <sim_demos> --out /content/eval_oracle --n 10

--oracle replaces the policy by a perfect one that reads the future poses from a recorded demo. It tests everything
around the policy (conversions, controller, gripper handling, success check) without any learning involved.
"""
import argparse, collections, json, math, os, sys, time
import numpy as np
from scipy.spatial.transform import Rotation as R, Slerp

HERE = os.path.dirname(os.path.abspath(__file__))
for p in (os.path.dirname(HERE), HERE):
    if p not in sys.path:
        sys.path.insert(0, p)

from sim_ctrl import MjKinematics, ik_step
from umi_sim.scene import CUP_RADIUS, CUP_HALF_H, SAUCER_HALF_H
from umi.common.pose_util import pose_to_mat, mat_to_pose, mat_to_pose10d, pose10d_to_mat
from diffusion_policy.common.pose_repr_util import convert_pose_mat_rep

TICK_S = 0.1                 # one policy tick = 5 control steps of 20 ms = the 10 Hz of the recorded demos
GRIPPER_MAX = 0.08
MIN_LIFT_Z = 0.08            # the cup counts as lifted above this height (same value the expert used)
DEFAULT_CLOSE_BELOW = 2 * CUP_RADIUS + 0.015


# ----------------------------------------------------------------------------------------------- conversions
def build_obs(history, shape_meta, start_pose, obs_pose_repr="relative"):
    """Policy input from the last few observations (oldest first). Mirrors get_real_umi_obs_dict of the UMI repo."""
    imgs = np.stack([h["camera0_rgb"] for h in history])
    out = {"camera0_rgb": np.moveaxis(imgs.astype(np.float32) / 255.0, -1, 1)}       # T,H,W,C -> T,C,H,W
    pos = np.stack([np.asarray(h["robot0_eef_pos"], np.float64).reshape(3) for h in history])
    rot = np.stack([np.asarray(h["robot0_eef_rot_axis_angle"], np.float64).reshape(3) for h in history])
    out["robot0_gripper_width"] = np.stack(
        [np.asarray(h["robot0_gripper_width"], np.float64).reshape(1) for h in history])
    pose_mat = pose_to_mat(np.concatenate([pos, rot], axis=-1))
    rel = convert_pose_mat_rep(pose_mat, base_pose_mat=pose_mat[-1], pose_rep=obs_pose_repr, backward=False)
    p10 = mat_to_pose10d(rel)
    out["robot0_eef_pos"] = p10[..., :3]
    out["robot0_eef_rot_axis_angle"] = p10[..., 3:]
    start_mat = pose_to_mat(np.asarray(start_pose, np.float64))
    rel_start = convert_pose_mat_rep(pose_mat, base_pose_mat=start_mat, pose_rep="relative", backward=False)
    out["robot0_eef_rot_axis_angle_wrt_start"] = mat_to_pose10d(rel_start)[..., 3:]
    keys = list(shape_meta["obs"].keys())
    missing = [k for k in keys if k not in out]
    if missing:
        raise KeyError("the policy expects observations %s that this simulator does not produce" % missing)
    return {k: np.ascontiguousarray(out[k], dtype=np.float32) for k in keys}


def actions_to_world(action, cur_pos, cur_rotvec, action_pose_repr="relative"):
    """(H,10) policy output -> world positions (H,3), rotation vectors (H,3), gripper widths (H,)."""
    action = np.asarray(action, np.float64)
    base = pose_to_mat(np.concatenate([np.asarray(cur_pos, np.float64).reshape(3),
                                       np.asarray(cur_rotvec, np.float64).reshape(3)]))
    mats = pose10d_to_mat(action[..., :9])
    mats = convert_pose_mat_rep(mats, base_pose_mat=base, pose_rep=action_pose_repr, backward=True)
    pose = mat_to_pose(mats)
    return pose[..., :3], pose[..., 3:], action[..., 9]


def grip_command(width, close_below=DEFAULT_CLOSE_BELOW):
    """The policy predicts the MEASURED finger width. While it holds the cup the fingers cannot get closer than the
    cup's diameter, so a prediction near that value means 'keep squeezing'. A real UMI gripper does this with a
    force-limited close; here every prediction below `close_below` becomes a full close command."""
    w = float(np.clip(width, 0.0, GRIPPER_MAX))
    return 0.0 if w < close_below else w


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - m) / d, (c + m) / d


# ----------------------------------------------------------------------------------------------- controller
class Controller:
    """Moves the TCP towards waypoints with IK + the arm's joint servos (same IK settings as the expert)."""

    def __init__(self, env, kin, q_home):
        self.env, self.kin, self.q_home = env, kin, np.asarray(q_home, float).copy()
        self.q_cmd = self.q_home.copy()
        pos, rot, _ = kin.fk(self.q_cmd)
        self.cmd_pos, self.cmd_rot = pos.copy(), R.from_matrix(rot)

    def go(self, pos, rot, grip, n_sub):
        """n_sub control steps from the previous command to (pos, rot); returns the observation after the last."""
        p0, r0 = self.cmd_pos, self.cmd_rot
        slerp = Slerp([0.0, 1.0], R.concatenate([r0, rot]))
        obs = None
        for j in range(1, n_sub + 1):
            a = j / n_sub
            p = p0 + a * (pos - p0)
            rm = slerp([a]).as_matrix()[0]
            self.q_cmd = ik_step(self.kin, self.q_cmd, p, rm, self.q_home)
            obs = self.env.step(self.q_cmd, grip, render=(j == n_sub))
        self.cmd_pos, self.cmd_rot = np.asarray(pos, float).copy(), rot
        return obs


# ----------------------------------------------------------------------------------------------- one episode
def cup_status(env):
    st = env.get_object_state()
    cup, sau = st["cup_pos"], st["saucer_pos"]
    dxy = float(np.linalg.norm(cup[:2] - sau[:2]))
    dz = float(cup[2] - (2 * SAUCER_HALF_H + CUP_HALF_H))
    up = float(env.data.xmat[env.cup_id].reshape(3, 3)[2, 2])
    return cup, dxy, dz, up


def run_episode(env, kin, act_fn, seed, shape_meta=None, steps_per_inference=8, max_time=40.0, obs_horizon=2,
                obs_pose_repr="relative", action_pose_repr="relative", close_below=DEFAULT_CLOSE_BELOW,
                success_hold_s=1.0, want_video=False, verbose=False):
    obs = env.reset(seed=seed)
    ctrl = Controller(env, kin, env.arm_q)
    hist = collections.deque([obs] * obs_horizon, maxlen=obs_horizon)
    start_pose = np.concatenate([obs["robot0_eef_pos"], obs["robot0_eef_rot_axis_angle"]])
    n_ticks = int(round(max_time / TICK_S))
    hold_need = int(round(success_hold_s / TICK_S))
    frames = []
    tick, held_ok, max_cup_z, infer_ms, n_infer = 0, 0, 0.0, [], 0
    outcome = None

    def snapshot():
        if want_video:
            frames.append(np.concatenate([env.render("front", 224, 224), hist[-1]["camera0_rgb"]], axis=1))

    snapshot()
    while tick < n_ticks and outcome is None:
        obs_np = build_obs(list(hist), shape_meta, start_pose, obs_pose_repr) if shape_meta is not None else None
        cur = hist[-1]
        t0 = time.time()
        act = np.asarray(act_fn(obs_np, {"tick": tick, "cur_pose": np.concatenate(
            [cur["robot0_eef_pos"], cur["robot0_eef_rot_axis_angle"]])}), np.float64)
        infer_ms.append(1000 * (time.time() - t0))
        n_infer += 1
        try:
            if not np.all(np.isfinite(act)):
                raise ValueError("non-finite action")
            pos, rotv, grip = actions_to_world(act, cur["robot0_eef_pos"], cur["robot0_eef_rot_axis_angle"], action_pose_repr)
            if not (np.all(np.isfinite(pos)) and np.all(np.isfinite(rotv))):
                raise ValueError("non-finite pose")
        except (ValueError, np.linalg.LinAlgError):
            outcome = ("nan_action", False)
            break
        for k in range(1, steps_per_inference + 1):
            k = min(k, len(pos) - 1)                         # action[0] is "now", so the plan starts at index 1
            obs = ctrl.go(pos[k], R.from_rotvec(rotv[k]), grip_command(grip[k], close_below), 5)
            hist.append(obs)
            tick += 1
            snapshot()
            cup, dxy, dz, up = cup_status(env)
            max_cup_z = max(max_cup_z, float(cup[2]))
            width = float(env.gripper_width)
            placed = dxy < 0.02 and abs(dz) < 0.012 and up > 0.95 and width > 0.06
            held_ok = held_ok + 1 if placed else 0
            if held_ok >= hold_need:
                outcome = ("success", True)
                break
            if cup[2] < -0.02:
                outcome = ("cup_fell_off_table", False)
                break
            if tick >= n_ticks:
                break
        if verbose:
            print("   tick %3d | cup z %.3f | grip %.3f" % (tick, max_cup_z, float(env.gripper_width)))
    cup, dxy, dz, up = cup_status(env)
    if outcome is None:
        if max_cup_z < MIN_LIFT_Z:
            outcome = ("never_lifted_cup", False)
        elif up < 0.95:
            outcome = ("cup_tipped_over", False)
        else:
            outcome = ("lifted_but_not_placed(dxy=%.3f)" % dxy, False)
    reason, ok = outcome
    return {"seed": int(seed), "success": bool(ok), "reason": reason, "ticks": tick, "sim_seconds": tick * TICK_S,
            "inferences": n_infer, "mean_infer_ms": float(np.mean(infer_ms)) if infer_ms else 0.0,
            "max_cup_z": float(max_cup_z), "final_dxy": float(dxy), "final_width": float(env.gripper_width)}, frames


# ----------------------------------------------------------------------------------------------- oracle
def make_oracle(demo_path, action_pose_repr="relative", horizon=16):
    with np.load(demo_path) as z:
        pos = np.asarray(z["robot0_eef_pos"], np.float64)
        rot = np.asarray(z["robot0_eef_rot_axis_angle"], np.float64)
        width = np.asarray(z["robot0_gripper_width"], np.float64).reshape(-1)
    mats = pose_to_mat(np.concatenate([pos, rot], axis=-1))
    T = len(mats)

    def fn(obs_np, info):
        idx = np.minimum(np.arange(info["tick"], info["tick"] + horizon), T - 1)
        base = pose_to_mat(info["cur_pose"])
        rel = convert_pose_mat_rep(mats[idx], base_pose_mat=base, pose_rep=action_pose_repr, backward=False)
        return np.concatenate([mat_to_pose10d(rel), width[idx][:, None]], axis=-1)

    return fn, T * TICK_S


# ----------------------------------------------------------------------------------------------- main
def write_video(path, frames, fps=10):
    import imageio
    try:
        with imageio.get_writer(path, fps=fps, codec="libx264", macro_block_size=1) as w:
            for f in frames:
                w.append_data(f)
        return path
    except Exception as e:                          # no ffmpeg: fall back to a gif
        gif = os.path.splitext(path)[0] + ".gif"
        imageio.mimsave(gif, frames, duration=1.0 / fps)
        print("   (mp4 failed: %s; wrote %s)" % (str(e).splitlines()[0][:80], gif))
        return gif


def load_policy(ckpt, device, num_inference_steps):
    import torch, dill, hydra
    from omegaconf import OmegaConf
    OmegaConf.register_new_resolver("eval", eval, replace=True)
    payload = torch.load(open(ckpt, "rb"), pickle_module=dill, map_location="cpu")
    cfg = payload["cfg"]
    ws = hydra.utils.get_class(cfg._target_)(cfg, output_dir="/content/_policy_tmp")
    ws.load_payload(payload, exclude_keys=["optimizer"])
    policy = ws.ema_model if cfg.training.use_ema else ws.model
    policy.num_inference_steps = num_inference_steps
    policy.eval().to(device)
    print("loaded %s | epoch %s | %s weights | encoder %s" % (
        os.path.basename(ckpt), getattr(ws, "epoch", "?"), "EMA" if cfg.training.use_ema else "raw",
        cfg.policy.obs_encoder.model_name))
    return policy, cfg


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--menagerie", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--ckpt")
    ap.add_argument("--oracle", action="store_true")
    ap.add_argument("--demo_dir")
    ap.add_argument("--n", type=int, default=10)
    ap.add_argument("--start_seed", type=int, default=5000)
    ap.add_argument("--steps_per_inference", type=int, default=8)
    ap.add_argument("--max_time", type=float, default=40.0)
    ap.add_argument("--num_inference_steps", type=int, default=16)
    ap.add_argument("--close_below", type=float, default=DEFAULT_CLOSE_BELOW)
    ap.add_argument("--videos", type=int, default=3)
    ap.add_argument("--verbose", action="store_true")
    args = ap.parse_args()
    if not args.oracle and not args.ckpt:
        ap.error("give --ckpt (or --oracle)")
    os.makedirs(args.out, exist_ok=True)

    from umi_sim.scene import build_scene
    from umi_sim.env import UmiSimEnv
    env = UmiSimEnv(build_scene(args.menagerie))
    kin = MjKinematics(env.model, env.tcp_id)

    policy = cfg = device = None
    shape_meta, obs_repr, act_repr = None, "relative", "relative"
    seeds, demos = list(range(args.start_seed, args.start_seed + args.n)), {}
    if args.oracle:
        files = sorted(f for f in os.listdir(args.demo_dir) if f.startswith("ep_") and f.endswith(".npz"))
        files = files[:args.n]
        assert files, "no ep_*.npz in " + str(args.demo_dir)
        seeds = [int(f[3:-4]) for f in files]
        demos = {s: os.path.join(args.demo_dir, f) for s, f in zip(seeds, files)}
    else:
        import torch
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        policy, cfg = load_policy(args.ckpt, device, args.num_inference_steps)
        shape_meta = cfg.task.shape_meta
        obs_repr, act_repr = cfg.task.pose_repr.obs_pose_repr, cfg.task.pose_repr.action_pose_repr
        print("pose representations: obs=%s action=%s | obs horizon %d" % (
            obs_repr, act_repr, shape_meta["obs"]["camera0_rgb"]["horizon"]))
    horizon = int(shape_meta["obs"]["camera0_rgb"]["horizon"]) if shape_meta is not None else 2

    results = []
    for i, seed in enumerate(seeds):
        max_time = args.max_time
        if args.oracle:
            act_fn, dur = make_oracle(demos[seed], act_repr)
            max_time = dur + 3.0
        else:
            import torch
            torch.manual_seed(seed)
            policy.reset()

            def act_fn(obs_np, info):
                with torch.no_grad():
                    ob = {k: torch.from_numpy(v).unsqueeze(0).to(device) for k, v in obs_np.items()}
                    return policy.predict_action(ob)["action_pred"][0].detach().cpu().numpy()
        t0 = time.time()
        res, frames = run_episode(env, kin, act_fn, seed, shape_meta=shape_meta,
                                  steps_per_inference=args.steps_per_inference, max_time=max_time,
                                  obs_horizon=horizon, obs_pose_repr=obs_repr, action_pose_repr=act_repr,
                                  close_below=args.close_below, want_video=(i < args.videos), verbose=args.verbose)
        res["wall_s"] = time.time() - t0
        if frames:
            res["video"] = os.path.basename(write_video(os.path.join(args.out, "ep_%05d.mp4" % seed), frames))
        results.append(res)
        print("seed %5d: %-34s %5.1f s sim | infer %5.0f ms | max cup z %.3f | %.0f s wall" % (
            seed, ("SUCCESS" if res["success"] else res["reason"]), res["sim_seconds"], res["mean_infer_ms"],
            res["max_cup_z"], res["wall_s"]), flush=True)
        json.dump({"results": results, "args": vars(args)}, open(os.path.join(args.out, "results.json"), "w"), indent=1)

    k = sum(r["success"] for r in results)
    lo, hi = wilson(k, len(results))
    kinds = collections.Counter(r["reason"].split("(")[0] for r in results)
    print("\nSUCCESS: %d / %d = %.0f %%  (95 %% interval %.0f-%.0f %%)" % (k, len(results), 100 * k / max(1, len(results)), 100 * lo, 100 * hi))
    print("outcomes:", dict(kinds))
    json.dump({"n": len(results), "successes": k, "rate": k / max(1, len(results)), "ci95": [lo, hi],
               "outcomes": dict(kinds)}, open(os.path.join(args.out, "summary.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

## 6. Check the scene loads and the evaluation script starts
A 1-episode oracle run without videos.

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH="$REPO:/content"
cd "$REPO"
"$PY" /content/umi_sim/run_policy.py --menagerie /content/mujoco_menagerie --oracle --demo_dir "$DEMO_DIR" --n 1 --videos 0 --out /content/eval_check 2>&1 | tail -8

## 7. Oracle test: a perfect policy through the whole chain (about 5 minutes)
Replays 10 recorded demos through the same conversion, controller and gripper rule the learned policy will use. The demo's own cup layout is recreated from its seed.

**Go on only if at least 8 of 10 succeed.** The scripted expert itself succeeded about 9 of 10 in Phase 3, so a perfect policy should be close to that. If not, the failure reasons tell what is wrong; paste them to me.

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH="$REPO:/content"
export WANDB_MODE=disabled
cd "$REPO"
"$PY" /content/umi_sim/run_policy.py --menagerie /content/mujoco_menagerie --oracle --demo_dir "$DEMO_DIR" --n 10 --videos 2 --out /content/eval_oracle 2>&1 | grep -v "^$"

In [ ]:
import json
s = json.load(open('/content/eval_oracle/summary.json'))
print('oracle success: %d / %d' % (s['successes'], s['n']), '| outcomes:', s['outcomes'])
print('ORACLE TEST:', 'PASS - go on to step 8' if s['successes'] >= 8 else 'FAIL - paste this output to Claude, do not evaluate the policy yet')

In [ ]:
from IPython.display import Video, display
import glob
for f in sorted(glob.glob('/content/eval_oracle/*.mp4'))[:1]:
    print(f, '(left: front camera, right: what the policy sees)')
    display(Video(f, embed=True, width=640))

## 8. Pick the checkpoint and run a quick policy test (10 episodes)
Uses `latest.ckpt` of the run named in `RUN_NAME` (the preset name from Phase 5). Test seeds start at 5000; the training demos used seeds 1000 and up, so these cup/saucer layouts were never seen in training. Videos of the first 4 episodes are saved. Expect roughly 20-40 s of wall time per episode (my estimate, unmeasured).

In [ ]:
RUN_NAME = 'resnet18'
CKPT = f'{ROOT}/checkpoints/{RUN_NAME}/checkpoints/latest.ckpt'
assert os.path.exists(CKPT), f'{CKPT} not found - see the checkpoint list in step 2'
os.environ['CKPT'] = CKPT
print('checkpoint:', CKPT, '| %.0f MB' % (os.path.getsize(CKPT) / 1e6))

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH="$REPO:/content"
export WANDB_MODE=disabled
cd "$REPO"
"$PY" /content/umi_sim/run_policy.py --menagerie /content/mujoco_menagerie --ckpt "$CKPT" --n 10 --start_seed 5000 --videos 4 --out /content/eval_quick 2>&1 | grep -v "^$"

In [ ]:
from IPython.display import Video, display
import glob
for f in sorted(glob.glob('/content/eval_quick/*.mp4'))[:4]:
    print(f)
    display(Video(f, embed=True, width=640))

**Reading the outcomes**

| outcome | what it usually means |
|---|---|
| `SUCCESS` | cup on saucer, upright, gripper open for 1 s |
| `never_lifted_cup` | the policy did not get the grasp: it missed the cup, hesitated, or closed too early or too late. Watch the video. |
| `lifted_but_not_placed` | it picked the cup up but did not put it on the saucer (wrong place, dropped, or ran out of time) |
| `cup_tipped_over` / `cup_fell_off_table` | a collision or a bad release |
| `nan_action` | the policy output something unusable; tell me |

A policy trained on 200 demos for 40 epochs is not expected to be perfect. 20-60 % success would already show the whole pipeline works; the numbers are what Phase 7 improves.

## 9. Full evaluation (50 episodes, about 20-30 minutes)
Results and videos are saved to Drive under `eval/`.

In [ ]:
N_EPISODES = 50
OUT = f'{ROOT}/eval/{RUN_NAME}_policy'
os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
os.environ['N_EPISODES'] = str(N_EPISODES)

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH="$REPO:/content"
export WANDB_MODE=disabled
cd "$REPO"
"$PY" /content/umi_sim/run_policy.py --menagerie /content/mujoco_menagerie --ckpt "$CKPT" --n $N_EPISODES --start_seed 6000 --videos 6 --out "$OUT" 2>&1 | grep -v "^$"

In [ ]:
import json, collections
r = json.load(open(f'{OUT}/results.json'))['results']
s = json.load(open(f'{OUT}/summary.json'))
print('success: %d / %d = %.0f %% (95%% interval %.0f-%.0f %%)' % (s['successes'], s['n'], 100 * s['rate'], 100 * s['ci95'][0], 100 * s['ci95'][1]))
print('outcomes:', s['outcomes'])
import numpy as np
print('mean policy inference: %.0f ms per call | mean episode length %.1f s' % (np.mean([x['mean_infer_ms'] for x in r]), np.mean([x['sim_seconds'] for x in r])))
ok = [x['sim_seconds'] for x in r if x['success']]
if ok:
    print('successful episodes took %.1f s on average (the scripted expert needs about 17 s)' % np.mean(ok))

## Phase 6 checklist

- [ ] Step 6: the 1-episode check ran
- [ ] Step 7: `ORACLE TEST: PASS` (this is the important one)
- [ ] Step 8: quick policy test ran; you watched at least 2 videos
- [ ] Step 9: full evaluation numbers saved to Drive

**Send me:** the step 7 summary line, the step 8 per-episode lines, a short description of what the failed videos show (does the arm go to the cup? grasp? carry? release?), and the step 9 summary.

Next: Phase 7 - evaluation and ablations (more demos, longer training, encoder choice, `steps_per_inference`, noise in the initial layout), driven by what the failures look like.